# 🔬 FSE 2026 Tutorial · Evaluating LLM-Generated Code Quality
## Section 02 · Maintainability & Reliability with SonarCloud

> **~30 minutes** Let's Get Started!!

### One-time setup
1. Go to **[sonarcloud.io](https://sonarcloud.io)** → Sign up free with GitHub/GitLab/Bitbucket
2. Create an **Organization** (e.g. `fse2026-yourname`) — choose the free plan
3. Go to **My Account → Access Tokens** → Make a new one and call it `fse2026`
4. In this Colab: click the 🔑 **Secrets** icon (left sidebar) → add secret named `SONARCLOUD_TOKEN` with your token value and turn on notebook access
5. In the cell following the "00c · Configure SonarCloud Token & Organization
" header edit the code to Set `SONARCLOUD_ORG` to the organization key you created on sonarcloud.io.

### Dataset
**[OpenAI HumanEval](https://github.com/openai/human-eval)** + frozen **[EvalPlus NeurIPS'23](https://github.com/evalplus/evalplus/releases/tag/v0.1.0)** solutions (temp=0.0 greedy decoding).

| Source | What it is |
|--------|------------|
| `human` | HumanEval canonical solutions (hand-written by OpenAI researchers) |
| `gpt4` | GPT-4 greedy solutions — EvalPlus v0.1.0 `gpt-4_temp_0.0` |
| `chatgpt` | ChatGPT (GPT-3.5) greedy solutions — EvalPlus v0.1.0 `chatgpt_temp_0.0` |
| `codellama` | CodeLlama-34B greedy solutions — EvalPlus v0.1.0 `code-llama-34b_temp_0.0` |
| `starcoder` | StarCoder greedy solutions — EvalPlus v0.1.0 `starcoder_temp_0.0` |
| `mistral` | Mistral-7B greedy solutions — EvalPlus v0.1.0 `mistral-7b_temp_0.0` |
| `codegen2b` | CodeGen-2B greedy solutions — EvalPlus v0.1.0 `codegen-2b_temp_0.0` |
| `codegen6b` | CodeGen-6B greedy solutions — EvalPlus v0.1.0 `codegen-6b_temp_0.0` |
| `codegen16b` | CodeGen-16B greedy solutions — EvalPlus v0.1.0 `codegen-16b_temp_0.0` |


### SonarCloud Metrics We Track
| Metric | Dimension | Meaning |
|--------|-----------|---------|
| `code_smells` | Maintainability | Overly complex methods, long param lists, duplicated blocks |
| `sqale_rating` | Maintainability | A–E (A = least tech debt) |
| `bugs` | Reliability | Null derefs, resource leaks, bad comparisons |
| `reliability_rating` | Reliability | A–E based on worst open bug |
| `cognitive_complexity` | Maintainability | How hard the code is to understand |

### Your 6 Tasks
| # | Task | Cell |
|---|------|------|
| 1 | Open Notebook Section 02 | you are here |
| 2 | Run SonarQube on the provided dataset | 02a |
| 3 | Extract code smell counts per source | 02b |
| 4 | Compare LLM vs. human distributions | 02c |
| 5 | Generate comparison bar chart | 02d |
| 6 | Note 2 surprising findings | 02e |

---
## 00 · Install Dependencies

In [ ]:
!pip install datasets requests pandas matplotlib seaborn --quiet
print('✅ Dependencies ready')

## 00b · Install sonar-scanner CLI


In [ ]:
import os, subprocess

SC_VER = '6.2.1.4610'
SC_DIR = f'/tmp/sonar-scanner-{SC_VER}-linux-x64'
SCANNER = f'{SC_DIR}/bin/sonar-scanner'

if not os.path.exists(SC_DIR):
    !wget -q 'https://binaries.sonarsource.com/Distribution/sonar-scanner-cli/sonar-scanner-cli-{SC_VER}-linux-x64.zip' -O /tmp/sc.zip
    !unzip -q /tmp/sc.zip -d /tmp/

ver = subprocess.check_output([SCANNER, '--version'], stderr=subprocess.STDOUT).decode().split('\n')[0]
print('✅', ver)

## 00c · Configure SonarCloud Token & Organization

> Add your token as a Colab Secret named `SONARCLOUD_TOKEN` (🔑 icon in left sidebar).
> Set `SONARCLOUD_ORG` below to the organization key you created on sonarcloud.io.

In [ ]:
from google.colab import userdata
import requests

# ── Your SonarCloud organization key ─────────────────────────────────
# Find it at sonarcloud.io → your org → Administration → Organization Key
SONARCLOUD_ORG = 'test1forfse'  # <── change this

# ── Load token from Colab Secrets ────────────────────────────────────
SONARCLOUD_TOKEN = userdata.get('SONARCLOUD_TOKEN')
SONARCLOUD_URL   = 'https://sonarcloud.io'
AUTH = (SONARCLOUD_TOKEN, '')

# Verify token works
r = requests.get(f'{SONARCLOUD_URL}/api/authentication/validate',
                 auth=AUTH, timeout=10)
valid = r.json().get('valid', False)
if valid:
    print('✅ Token valid — connected to SonarCloud')
else:
    raise ValueError('❌ Token invalid — check your SONARCLOUD_TOKEN secret')

---
## 01 · Load Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

he = load_dataset('openai/openai_humaneval', split='test')
print(f'HumanEval: {len(he)} problems')

human_by_id  = {}
prompt_by_id = {}
for row in he:
    human_by_id[row['task_id']] = {
        'task_id': row['task_id'],
        'entry':   row['entry_point'],
        'code':    row['prompt'] + row['canonical_solution'],
    }
    prompt_by_id[row['task_id']] = row['prompt']
print(f'✅ {len(human_by_id)} human solutions loaded')

In [ ]:
import json, zipfile, io, re

EVALPLUS_URLS = {
    'gpt4':       'https://github.com/evalplus/evalplus/releases/download/v0.1.0/gpt-4_temp_0.0.zip',
    'chatgpt':    'https://github.com/evalplus/evalplus/releases/download/v0.1.0/chatgpt_temp_0.0.zip',
    'codellama':  'https://github.com/evalplus/evalplus/releases/download/v0.1.0/code-llama-34b_temp_0.0.zip',
    'starcoder':  'https://github.com/evalplus/evalplus/releases/download/v0.1.0/starcoder_temp_0.0.zip',
    'mistral':    'https://github.com/evalplus/evalplus/releases/download/v0.1.0/mistral-7b_temp_0.0.zip',
    'codegen2b':  'https://github.com/evalplus/evalplus/releases/download/v0.1.0/codegen-2b_temp_0.0.zip',
    'codegen6b':  'https://github.com/evalplus/evalplus/releases/download/v0.1.0/codegen-6b_temp_0.0.zip',
    'codegen16b': 'https://github.com/evalplus/evalplus/releases/download/v0.1.0/codegen-16b_temp_0.0.zip',
}

def load_evalplus_samples(url):
    print(f'  Downloading {url.split("/")[-1]}...')
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    samples = {}
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for name in zf.namelist():
            m = re.search(r'HumanEval_(\d+)/0\.py$', name)
            if m:
                task_id = f'HumanEval/{m.group(1)}'
                with zf.open(name) as f:
                    samples[task_id] = f.read().decode('utf-8')
    print(f'  ✅ {len(samples)} solutions')
    return samples

print('Loading frozen LLM solutions...')
llm_sols = {}
for key, url in EVALPLUS_URLS.items():
    llm_sols[key] = load_evalplus_samples(url)

In [ ]:
print("ORG:", repr(SONARCLOUD_ORG))
print("TOKEN EXISTS:", SONARCLOUD_TOKEN is not None)
print("AUTH USER:", AUTH[0][:5] if AUTH[0] else None)

In [ ]:
import random
random.seed(2026)

SOURCES = ['human', 'gpt4', 'chatgpt', 'codellama', 'starcoder',
           'mistral', 'codegen2b', 'codegen6b', 'codegen16b']

SOURCE_LABELS = {
    'human':      'Human',
    'gpt4':       'GPT-4',
    'chatgpt':    'ChatGPT',
    'codellama':  'CodeLlama-34B',
    'starcoder':  'StarCoder',
    'mistral':    'Mistral-7B',
    'codegen2b':  'CodeGen-2B',
    'codegen6b':  'CodeGen-6B',
    'codegen16b': 'CodeGen-16B',
}

COLORS = {
    'human':      '#2ecc71',
    'gpt4':       '#1a5276',
    'chatgpt':    '#3498db',
    'codellama':  '#e67e22',
    'starcoder':  '#e74c3c',
    'mistral':    '#8e44ad',
    'codegen2b':  '#f0b27a',
    'codegen6b':  '#ca6f1e',
    'codegen16b': '#784212',
}

common_ids = set(human_by_id.keys())
for key, sols in llm_sols.items():
    common_ids &= set(sols.keys())
sample_ids = sorted(common_ids)

def build_full_code(task_id, code):
    stripped = code.lstrip()
    if stripped.startswith('def ') or stripped.startswith('import '):
        return code
    return prompt_by_id[task_id] + code

records = []
for tid in sample_ids:
    entry = human_by_id[tid]['entry']
    records.append({'task_id': tid, 'entry': entry, 'source': 'human',
                    'code': human_by_id[tid]['code']})
    for key, sols in llm_sols.items():
        records.append({'task_id': tid, 'entry': entry, 'source': key,
                        'code': build_full_code(tid, sols[tid])})

df = pd.DataFrame(records)
print(f'✅ {len(df)} snippets | {df["task_id"].nunique()} tasks x {df["source"].nunique()} sources')

## 02a · Task 2 - Run SonarQube on the provided dataset

In [ ]:
# 02a — Write snippets to temp disk, create SonarCloud projects, run scanner
import time

PROJECT_KEYS = {
    s: f'{SONARCLOUD_ORG}-humaneval-{s}'.lower()
    for s in SOURCES
}
PROJ_ROOT    = '/tmp/sonar_projects'

for source in SOURCES:
    src_dir = f'{PROJ_ROOT}/{source}/src'
    os.makedirs(src_dir, exist_ok=True)
    for _, row in df[df['source'] == source].iterrows():
        safe_name = row['task_id'].replace('/', '_')
        with open(f'{src_dir}/{safe_name}.py', 'w') as f:
            f.write(row['code'])
print(f'✅ {len(df)} .py files written')

for source, key in PROJECT_KEYS.items():
    r = requests.post(
        f'{SONARCLOUD_URL}/api/projects/create', auth=AUTH,
        data={'name': f'HumanEval - {SOURCE_LABELS[source]}',
              'project': key, 'organization': SONARCLOUD_ORG, 'visibility': 'public'}
    )
    status = 'created' if r.status_code in (200, 201) else (
        'exists' if 'already' in r.text else r.text[:50]
    )
    print(f'  [{key}] {status}')

print('\n🔍 Running sonar-scanner...')
for source in SOURCES:
    key      = PROJECT_KEYS[source]
    proj_dir = f'{PROJ_ROOT}/{source}'
    props = '\n'.join([
        f'sonar.projectKey={key}',
        f'sonar.projectName=HumanEval - {SOURCE_LABELS[source]}',
        f'sonar.organization={SONARCLOUD_ORG}',
        'sonar.sources=src',
        'sonar.language=py',
        'sonar.python.version=3',
        f'sonar.host.url={SONARCLOUD_URL}',
        f'sonar.token={SONARCLOUD_TOKEN}',
        'sonar.scm.disabled=true',
    ])
    with open(f'{proj_dir}/sonar-project.properties', 'w') as f:
        f.write(props)
    result = subprocess.run(
        [SCANNER, f'-Dproject.settings={proj_dir}/sonar-project.properties'],
        cwd=proj_dir, capture_output=True, text=True, timeout=180
    )
    ok = 'EXECUTION SUCCESS' in result.stdout
    print(f'  {SOURCE_LABELS[source]:>15}: {"✅ success" if ok else "❌ FAILED"}')
    if not ok:
      print("STDOUT:")
      print(result.stdout[-2000:])
      print("STDERR:")
      print(result.stderr[-2000:])

print('\n⏳ Waiting 20s for SonarCloud to process...')
time.sleep(20)
print('✅ Ready.')

## 02b - Task 3 · Extract Code Smell Counts Per Source

In [ ]:
METRICS = ['code_smells', 'sqale_index', 'sqale_rating', 'bugs',
           'reliability_rating', 'cognitive_complexity', 'ncloc']
RATING_MAP = {'1': 'A', '2': 'B', '3': 'C', '4': 'D', '5': 'E'}

def fetch_project_metrics(key):
    r = requests.get(
        f'{SONARCLOUD_URL}/api/measures/component', auth=AUTH,
        params={'component': key, 'metricKeys': ','.join(METRICS)}
    )
    return {m['metric']: m.get('value', '0')
            for m in r.json().get('component', {}).get('measures', [])}

def fetch_file_metrics(key):
    all_rows = []
    page = 1
    while True:
        r = requests.get(
            f'{SONARCLOUD_URL}/api/measures/component_tree', auth=AUTH,
            params={'component': key,
                    'metricKeys': 'code_smells,bugs,cognitive_complexity',
                    'qualifiers': 'FIL', 'ps': 100, 'p': page}
        )
        data = r.json()
        for comp in data.get('components', []):
            row = {'file': comp['name'].replace('.py', '')}
            for m in comp.get('measures', []):
                row[m['metric']] = int(m.get('value', 0))
            all_rows.append(row)
        total = data.get('paging', {}).get('total', 0)
        if page * 100 >= total:
            break
        page += 1
    return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()

proj_rows = []
for source in SOURCES:
    m = fetch_project_metrics(PROJECT_KEYS[source])
    m['source'] = source
    proj_rows.append(m)

df_proj = pd.DataFrame(proj_rows)
for col in ['code_smells', 'sqale_index', 'bugs', 'cognitive_complexity', 'ncloc']:
    if col in df_proj.columns:
        df_proj[col] = pd.to_numeric(df_proj[col], errors='coerce').fillna(0).astype(int)
for rcol in ['sqale_rating', 'reliability_rating']:
    if rcol in df_proj.columns:
        df_proj[rcol + '_label'] = df_proj[rcol].apply(
            lambda v: RATING_MAP.get(str(int(float(v))), '?') if v and v != '0' else '?'
        )
df_proj['smells_per_100loc'] = (df_proj['code_smells'] / df_proj['ncloc'] * 100).round(1)
df_proj['source_label']      = df_proj['source'].map(SOURCE_LABELS)

DISPLAY = [c for c in ['source_label', 'ncloc', 'code_smells', 'smells_per_100loc',
           'sqale_rating_label', 'bugs', 'reliability_rating_label', 'cognitive_complexity']
           if c in df_proj.columns]
print('📊 Project-Level SonarCloud Metrics')
print('=' * 75)
print(df_proj[DISPLAY].to_string(index=False))

## 02c - Task 4 · Compare LLM vs. Human Distributions

In [ ]:
file_dfs = []
for source in SOURCES:
    fdf = fetch_file_metrics(PROJECT_KEYS[source])
    if not fdf.empty:
        fdf['source']       = source
        fdf['source_label'] = SOURCE_LABELS[source]
        file_dfs.append(fdf)

df_files = pd.concat(file_dfs, ignore_index=True)
for col in ['code_smells', 'bugs', 'cognitive_complexity']:
    df_files[col] = pd.to_numeric(df_files[col], errors='coerce').fillna(0).astype(int)

for metric in ['code_smells', 'cognitive_complexity', 'bugs']:
    print(f'\n📋 {metric}')
    print(df_files.groupby('source_label')[metric].describe().round(2))

## 02d - Task 5 · Generate Comparison Bar Chart

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

# Ordered for a logical narrative: proprietary → open large → open small
SOURCE_ORDER = ['human', 'gpt4', 'chatgpt', 'codellama', 'starcoder',
                'mistral', 'codegen2b', 'codegen6b', 'codegen16b']

df_plot = df_proj.set_index('source').loc[SOURCE_ORDER].reset_index()
labels  = [SOURCE_LABELS[s] for s in SOURCE_ORDER]
colors  = [COLORS[s] for s in SOURCE_ORDER]

def bar_panel(ax, values, labels, colors, title, ylabel, hline=None, hline_label=None):
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=1.5, zorder=3)
    ax.set_title(title, fontweight='bold', fontsize=10, pad=8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.annotate('lower = better', xy=(0.98, 0.97), xycoords='axes fraction',
                fontsize=8, color='gray', ha='right', va='top')
    if hline is not None:
        ax.axhline(hline, color='red', linestyle='--', alpha=0.55, linewidth=1.2, label=hline_label)
        ax.legend(fontsize=8)
    vmax = max(values) if max(values) > 0 else 1
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + vmax * 0.02,
                str(int(val)), ha='center', va='bottom', fontweight='bold', fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(
    f'SonarCloud: {len(SOURCE_ORDER)} sources · OpenAI HumanEval · '
    f'{df["task_id"].nunique()} tasks · temp=0.0',
    fontsize=13, fontweight='bold', y=1.01
)
bar_panel(axes[0], df_plot['code_smells'].tolist(), labels, colors,
          'Code Smells\n(Maintainability)', 'Total Smell Count')
bar_panel(axes[1], df_plot['bugs'].tolist(), labels, colors,
          'Bugs Detected\n(Reliability)', 'Bug Count')
bar_panel(axes[2], df_plot['cognitive_complexity'].tolist(), labels, colors,
          'Cognitive Complexity\n(Maintainability)', 'Total Cognitive Complexity',
          hline=250, hline_label='15/fn × 164 tasks')
patches = [mpatches.Patch(color=COLORS[s], label=SOURCE_LABELS[s]) for s in SOURCE_ORDER]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.savefig('/tmp/sonarcloud_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved.')

In [ ]:
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig2.suptitle('Smell Density and Per-Task Distribution', fontsize=12, fontweight='bold')

bars = ax1.bar(labels, df_plot['smells_per_100loc'].tolist(),
               color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Code Smell Density (per 100 LOC)\nNormalized for fair comparison', fontweight='bold')
ax1.set_ylabel('Smells per 100 Lines')
ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, df_plot['smells_per_100loc']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

order   = [SOURCE_LABELS[s] for s in SOURCE_ORDER]
palette = {SOURCE_LABELS[s]: COLORS[s] for s in SOURCE_ORDER}
sns.stripplot(data=df_files, x='source_label', y='cognitive_complexity',
              order=order, palette=palette, ax=ax2, jitter=True, alpha=0.4, size=3)
for i, src_label in enumerate(order):
    mean_val = df_files[df_files['source_label'] == src_label]['cognitive_complexity'].mean()
    ax2.hlines(mean_val, i - 0.35, i + 0.35, colors='black', linewidth=2,
               label='mean' if i == 0 else '')
ax2.legend(['mean'], fontsize=9)
ax2.set_title(f'Cognitive Complexity per Task\n{df["task_id"].nunique()} problems — each dot = 1 task',
              fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('Cognitive Complexity')
ax2.set_xticklabels(order, rotation=35, ha='right', fontsize=8)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/sonarcloud_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 02e - Task 6 · Note 2 Surprising Findings


In [ ]:
dp = df_proj.set_index('source')

most_smells   = dp['code_smells'].idxmax()
fewest_smells = dp['code_smells'].idxmin()
most_complex  = dp['cognitive_complexity'].idxmax()
least_complex = dp['cognitive_complexity'].idxmin()

print('=' * 65)
print('FINDING #1 — Smells and Bugs Do Not Move Together')
print('=' * 65)
print(f'  {SOURCE_LABELS[most_smells]} has the most smells ({dp["code_smells"][most_smells]}),')
print(f'  yet all models have near-zero bugs ({dp["bugs"].to_dict()}).')
print('  LLMs add defensive guards SonarCloud flags as smells,')
print('  but those same guards prevent bugs.')
print('  => High smell count != unreliable code.')
print()
print('=' * 65)
print('FINDING #2 — Model Size Does Not Guarantee Lower Complexity')
print('=' * 65)
cg = {s: dp['cognitive_complexity'][s] for s in ['codegen2b','codegen6b','codegen16b']}
for k, v in cg.items():
    print(f'  {SOURCE_LABELS[k]:>15}: cognitive complexity = {v}')
print()
print(f'  Most complex overall: {SOURCE_LABELS[most_complex]} ({dp["cognitive_complexity"][most_complex]})')
print(f'  Least complex overall: {SOURCE_LABELS[least_complex]} ({dp["cognitive_complexity"][least_complex]})')
print('  => Bigger models write more elaborate code, not simpler code.')
print()
print('=' * 65)
print('DISCUSSION: Does higher smell density always mean lower quality?')
print('When might SonarCloud smells be misleading?')
print('=' * 65)

In [ ]:
#Fill in your findings here!!

YOUR_FINDING_1 = 'Finding #1: ...'
YOUR_FINDING_2 = 'Finding #2: ...'
print(YOUR_FINDING_1)
print(YOUR_FINDING_2)

##Full Results Table & SonarCloud Dashboard

In [ ]:
ICONS = {'A': '🟢', 'B': '🟡', 'C': '🟠', 'D': '🔴', 'E': '⛔'}
print('=' * 78)
print('SONARCLOUD SUMMARY — Maintainability & Reliability')
print('=' * 78)
print(f"{'Source':<18} {'LOC':>5} {'Smells':>7} {'/100L':>6} {'Maint.':>9} {'Bugs':>5} {'Reliab.':>9} {'Cog.Cplx':>9}")
print('-' * 78)
for src in SOURCE_ORDER:
    row = dp.loc[src]
    ml  = row.get('sqale_rating_label', '?')
    rl  = row.get('reliability_rating_label', '?')
    print(
        f"{SOURCE_LABELS[src]:<18}"
        f"{int(row.get('ncloc', 0)):>5}"
        f"{int(row.get('code_smells', 0)):>7}"
        f"{float(row.get('smells_per_100loc', 0)):>6.1f}"
        f"  {ICONS.get(ml, '⚪')}{ml:>5}"
        f"{int(row.get('bugs', 0)):>5}"
        f"  {ICONS.get(rl, '⚪')}{rl:>5}"
        f"{int(row.get('cognitive_complexity', 0)):>9}"
    )
print('=' * 78)
print('A=best, E=worst')
print(f'\nBrowse full results at {SONARCLOUD_URL}/organizations/{SONARCLOUD_ORG}/projects')
for src, key in PROJECT_KEYS.items():
    print(f'  {SOURCE_LABELS[src]:>18}: {SONARCLOUD_URL}/project/overview?id={key}')

# Section 03 · Security Vulnerability Analysis with Bandit

> ** ~30 minutes** | Continues from Section 02 (Section 2 is a Prerequisite!!)

### Key Finding from the Literature
LLM-generated code often appears syntactically correct yet harbors exploitable vulnerabilities that escape casual review.

### Common Vulnerability Types We Look For and Encounter
| CWE | Type | Bandit Pattern |
|-----|------|---------------|
| CWE-78 | OS Command Injection | `subprocess` with unsanitized input |
| CWE-89 | SQL Injection | string-formatted SQL queries |
| CWE-22 | Path Traversal | `open()` with user-controlled paths |
| CWE-798 | Hardcoded Credentials | API keys embedded in source |
| CWE-502 | Insecure Deserialization | `pickle.loads()` on untrusted data |
| CWE-330 | Weak Randomness | `random.random()` for tokens |

### Bandit Severity × Confidence Matrix
| | LOW conf. | MED conf. | HIGH conf. |
|--|-----------|-----------|------------|
| **HIGH sev.** | Medium | High | **Critical** |
| **MED sev.** | Low | Medium | High |
| **LOW sev.** | Informational | Low | Medium |

### Your 6 Tasks
| # | Task | Cell |
|---|------|------|
| 1 | Open Notebook Section 03 | you are here |
| 2 | Run Bandit on the full dataset | 03a |
| 3 | Filter HIGH severity findings | 03b |
| 4 | Identify top 3 vulnerability types | 03c |
| 5 | Compare LLM vs. human profiles | 03d |
| 6 | Flag at least one false positive | 03e |

---
## 00a - Install Bandit

In [ ]:
!pip install bandit --quiet
!bandit --version

---
## 00b - Write Dataset to Temp Disk

In [ ]:
import os, json, subprocess, pandas as pd
from collections import Counter

BANDIT_ROOT = '/tmp/bandit_projects'

for source in SOURCES:
    src_dir = f'{BANDIT_ROOT}/{source}'
    os.makedirs(src_dir, exist_ok=True)
    for _, row in df[df['source'] == source].iterrows():
        safe_name = row['task_id'].replace('/', '_')
        with open(f'{src_dir}/{safe_name}.py', 'w') as f:
            f.write(row['code'])

print(f'✅ Written {len(df)} files across {len(SOURCES)} sources to {BANDIT_ROOT}')

---
## 03a · Task 2 — Run Bandit on the Full Dataset

> Runs `bandit -r <dir> -f json` on each source group. Matches the slides pipeline: `bandit -r ./snippets -f json -o report.json`

In [ ]:
bandit_results = {}

for source in SOURCES:
    src_dir = f'{BANDIT_ROOT}/{source}'
    out_file = f'{BANDIT_ROOT}/{source}_report.json'

    result = subprocess.run(
        ['bandit', '-r', src_dir, '-f', 'json', '-o', out_file, '-q'],
        capture_output=True, text=True
    )

    try:
        with open(out_file) as f:
            data = json.load(f)
        issues = data.get('results', [])
        bandit_results[source] = issues
        print(f'  {SOURCE_LABELS[source]:>18}: {len(issues):>4} issues found')
    except Exception as e:
        bandit_results[source] = []
        print(f'  {SOURCE_LABELS[source]:>18}: ERROR — {e}')

total = sum(len(v) for v in bandit_results.values())
print(f'\n✅ Total issues found across all sources: {total}')

---
## 03b · Task 3 — Filter HIGH Severity Findings

In [ ]:
# @title
# Build a flat dataframe of all issues with CWE mapping

BANDIT_TO_CWE = {
    'B101': {'cwe': 'CWE-617', 'name': 'Assert Used',           'description': 'assert statements removed in optimised bytecode'},
    'B102': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'exec() with unsanitized input'},
    'B103': {'cwe': 'CWE-732', 'name': 'Insecure File Perms',   'description': 'chmod setting overly permissive modes'},
    'B105': {'cwe': 'CWE-798', 'name': 'Hardcoded Password',    'description': 'password-like string assigned to variable'},
    'B106': {'cwe': 'CWE-798', 'name': 'Hardcoded Password',    'description': 'password passed as function argument'},
    'B107': {'cwe': 'CWE-798', 'name': 'Hardcoded Password',    'description': 'password in function default argument'},
    'B108': {'cwe': 'CWE-377', 'name': 'Insecure Temp File',    'description': 'probable insecure use of temp file'},
    'B110': {'cwe': 'CWE-390', 'name': 'Try/Except Pass',       'description': 'except pass silently swallows exceptions'},
    'B112': {'cwe': 'CWE-390', 'name': 'Try/Except Continue',   'description': 'except continue silently swallows exceptions'},
    'B201': {'cwe': 'CWE-094', 'name': 'Flask Debug Mode',      'description': 'Flask app run with debug=True'},
    'B301': {'cwe': 'CWE-502', 'name': 'Insecure Deserialize',  'description': 'pickle.loads() on untrusted data'},
    'B302': {'cwe': 'CWE-502', 'name': 'Insecure Deserialize',  'description': 'marshal.loads() on untrusted data'},
    'B303': {'cwe': 'CWE-327', 'name': 'Weak Cryptography',     'description': 'MD5/SHA1 used for cryptographic purpose'},
    'B304': {'cwe': 'CWE-327', 'name': 'Weak Cipher',           'description': 'DES/RC2/RC4/RC5 cipher used'},
    'B305': {'cwe': 'CWE-327', 'name': 'Weak Cipher Mode',      'description': 'ECB cipher mode used'},
    'B307': {'cwe': 'CWE-094', 'name': 'Code Injection',        'description': 'eval() used — arbitrary code execution risk'},
    'B310': {'cwe': 'CWE-601', 'name': 'URL Redirect',          'description': 'urllib open with user-supplied URL'},
    'B311': {'cwe': 'CWE-330', 'name': 'Weak Randomness',       'description': 'random module not suitable for security use'},
    'B314': {'cwe': 'CWE-611', 'name': 'XML Injection',         'description': 'xml.etree.ElementTree vulnerable to XML attacks'},
    'B320': {'cwe': 'CWE-611', 'name': 'XML Injection',         'description': 'lxml.etree vulnerable to XML attacks'},
    'B324': {'cwe': 'CWE-327', 'name': 'Weak Hash (MD5/SHA1)',  'description': 'hashlib MD5/SHA1 flagged as weak for security'},
    'B501': {'cwe': 'CWE-295', 'name': 'TLS Verify Disabled',   'description': 'SSL/TLS certificate verification disabled'},
    'B502': {'cwe': 'CWE-326', 'name': 'Weak TLS Version',      'description': 'SSLv2 or SSLv3 explicitly allowed'},
    'B506': {'cwe': 'CWE-20',  'name': 'YAML Load',             'description': 'yaml.load() without safe Loader'},
    'B601': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'paramiko exec_command with shell=True'},
    'B602': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'subprocess with shell=True'},
    'B603': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'subprocess without shell=True (lower risk)'},
    'B604': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'function call with shell=True'},
    'B605': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'os.system() used'},
    'B606': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'os.popen() used'},
    'B607': {'cwe': 'CWE-78',  'name': 'OS Command Injection',  'description': 'start process with partial executable path'},
    'B608': {'cwe': 'CWE-89',  'name': 'SQL Injection',         'description': 'string-formatted SQL query'},
    'B701': {'cwe': 'CWE-94',  'name': 'Template Injection',    'description': 'jinja2 autoescape disabled'},
    'B703': {'cwe': 'CWE-94',  'name': 'Template Injection',    'description': 'Django mark_safe() used'},
}

In [ ]:

rows = []
for source, issues in bandit_results.items():
    for issue in issues:
        tid = issue.get('test_id', '')
        cwe_info = BANDIT_TO_CWE.get(tid, {
            'cwe': 'CWE-???', 'name': tid, 'description': issue.get('test_name', '')
        })
        rows.append({
            'source':        source,
            'source_label':  SOURCE_LABELS[source],
            'file':          os.path.basename(issue.get('filename', '')).replace('.py', ''),
            'test_id':       tid,
            'cwe':           cwe_info['cwe'],
            'vuln_name':     cwe_info['name'],
            'description':   cwe_info['description'],
            'severity':      issue.get('issue_severity', ''),
            'confidence':    issue.get('issue_confidence', ''),
            'line':          issue.get('line_number', 0),
            'text':          issue.get('issue_text', ''),
        })

df_issues = pd.DataFrame(rows)

SEV_SCORE  = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
CONF_SCORE = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
df_issues['priority_score'] = (
    df_issues['severity'].map(SEV_SCORE).fillna(0) *
    df_issues['confidence'].map(CONF_SCORE).fillna(0)
)
df_issues['priority'] = df_issues['priority_score'].map(
    {9: 'Critical', 6: 'High', 4: 'High', 3: 'Medium', 2: 'Medium', 1: 'Low', 0: 'Info'}
)

df_high = df_issues[df_issues['severity'] == 'HIGH'].copy()

print(f'Total issues:         {len(df_issues)}')
print(f'HIGH severity issues: {len(df_high)}')
print()
print('HIGH severity by source (with CWE):')
print(df_high.groupby(['source_label', 'cwe', 'vuln_name']).size().to_string())

---
## 03c · Task 4 — Identify Top 3 Vulnerability Types

In [ ]:
import matplotlib.pyplot as plt

# Top vulnerability types across all sources
vuln_counts = df_issues.groupby(['test_id', 'cwe', 'vuln_name']).size().sort_values(ascending=False)
print('Top 10 vulnerability types (all sources):')
print(vuln_counts.head(10).to_string())
print()

top3 = vuln_counts.head(3)
top3_ids   = top3.index.get_level_values('test_id').tolist()
top3_cwes  = top3.index.get_level_values('cwe').tolist()
top3_names = top3.index.get_level_values('vuln_name').tolist()

print('Top 3:')
for tid, cwe, name, count in zip(top3_ids, top3_cwes, top3_names, top3.values):
    print(f'  {tid} ({cwe}): {name} — {count} occurrences')

# Severity breakdown
print()
print('Severity breakdown for top 3:')
top3_df = df_issues[df_issues['test_id'].isin(top3_ids)]
print(top3_df.groupby(['test_id', 'cwe', 'vuln_name', 'severity']).size().to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
top10 = vuln_counts.head(10)
labels_v = [f"{tid}\n{cwe}\n{name[:25]}" for tid, cwe, name in top10.index]
bars = ax.bar(labels_v, top10.values, color='#c0392b', edgecolor='white', linewidth=1.2)
ax.set_title('Bandit Vulnerability Types\nAll sources combined',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Issue Count')
ax.set_xticklabels(labels_v, fontsize=7)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, top10.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/bandit_top_vulns.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 03d · Task 5 — Compare LLM vs. Human Vulnerability Profiles

In [ ]:
SOURCE_ORDER = ['human', 'gpt4', 'chatgpt', 'codellama', 'starcoder',
                'mistral', 'codegen2b', 'codegen6b', 'codegen16b']

# Issues per 100 LOC per source (normalized)
# Get LOC from df
loc_by_source = df.groupby('source')['code'].apply(lambda x: x.str.count('\n').sum())

summary_rows = []
for source in SOURCE_ORDER:
    issues     = bandit_results.get(source, [])
    total      = len(issues)
    high_sev   = sum(1 for i in issues if i.get('issue_severity') == 'HIGH')
    med_sev    = sum(1 for i in issues if i.get('issue_severity') == 'MEDIUM')
    low_sev    = sum(1 for i in issues if i.get('issue_severity') == 'LOW')
    critical   = sum(1 for i in issues
                     if i.get('issue_severity') == 'HIGH'
                     and i.get('issue_confidence') == 'HIGH')
    loc        = int(loc_by_source.get(source, 1))
    per_100    = round(total / loc * 100, 2) if loc > 0 else 0
    summary_rows.append({
        'source':        source,
        'source_label':  SOURCE_LABELS[source],
        'total_issues':  total,
        'high_sev':      high_sev,
        'med_sev':       med_sev,
        'low_sev':       low_sev,
        'critical':      critical,
        'loc':           loc,
        'per_100loc':    per_100,
    })

df_summary = pd.DataFrame(summary_rows)
DISP = ['source_label','total_issues','high_sev','med_sev','low_sev','critical','loc','per_100loc']
print('📊 Security Vulnerability Summary')
print('=' * 80)
print(df_summary[DISP].to_string(index=False))

# Stacked bar: HIGH / MED / LOW per source
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Bandit Security Analysis: LLM vs. Human\nOpenAI HumanEval · All Sources',
             fontsize=13, fontweight='bold')

df_plot_s = df_summary.set_index('source').loc[SOURCE_ORDER].reset_index()
src_labels = [SOURCE_LABELS[s] for s in SOURCE_ORDER]
colors_s   = [COLORS[s] for s in SOURCE_ORDER]
x = range(len(SOURCE_ORDER))

# Left: stacked bar by severity
b_low  = ax1.bar(x, df_plot_s['low_sev'],  color='#f9e79f', edgecolor='white', label='LOW')
b_med  = ax1.bar(x, df_plot_s['med_sev'],  bottom=df_plot_s['low_sev'],
                 color='#f0b27a', edgecolor='white', label='MEDIUM')
b_high = ax1.bar(x, df_plot_s['high_sev'],
                 bottom=df_plot_s['low_sev']+df_plot_s['med_sev'],
                 color='#c0392b', edgecolor='white', label='HIGH')
ax1.set_title('Issues by Severity', fontweight='bold')
ax1.set_ylabel('Issue Count')
ax1.set_xticks(list(x))
ax1.set_xticklabels(src_labels, rotation=35, ha='right', fontsize=8)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Right: issues per 100 LOC (normalized)
bars2 = ax2.bar(x, df_plot_s['per_100loc'], color=colors_s, edgecolor='white', linewidth=1.2)
ax2.set_title('Issue Density (per 100 LOC)\nNormalized', fontweight='bold')
ax2.set_ylabel('Issues per 100 Lines')
ax2.set_xticks(list(x))
ax2.set_xticklabels(src_labels, rotation=35, ha='right', fontsize=8)
ax2.grid(axis='y', alpha=0.3)
ax2.annotate('lower = better', xy=(0.98, 0.97), xycoords='axes fraction',
             fontsize=8, color='gray', ha='right', va='top')
for bar, val in zip(bars2, df_plot_s['per_100loc']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/bandit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved.')

In [ ]:
import seaborn as sns

# Heatmap: vulnerability type × source
pivot = df_issues.groupby(['cwe', 'source_label']).size().unstack(fill_value=0)
top10_cwes = df_issues['cwe'].value_counts().head(10).index
pivot_top  = pivot.loc[pivot.index.isin(top10_cwes)]
col_order  = [SOURCE_LABELS[s] for s in SOURCE_ORDER if SOURCE_LABELS[s] in pivot_top.columns]
pivot_top  = pivot_top[col_order]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_top, annot=True, fmt='d', cmap='Reds',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Issue Count'})
ax.set_title('Vulnerability Type × Source Heatmap\n(CWE IDs)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('CWE ID')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/bandit_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 03e · Task 6 — Flag at Least One False Positive

Bandit does static analysis — it flags *patterns*, not always proven vulnerabilities. There is a possibility that some findings can be false positives. Your job: look at a flagged line, read the code, and decide if it's a real issue.

In [ ]:
# ── Task 6 · Flag at least one false positive ──────────────────────────────
#
# Below are 3 real Bandit findings from this dataset.
# At least one is a FALSE POSITIVE — Bandit is wrong.
# Read the CWE definition, the task prompt, and the flagged code
# for each candidate, then record your verdict in the next cell.

CWE_DEFINITIONS = {
    'CWE-094': ('Code Injection',
                'The product constructs all or part of a code segment using externally-influenced input from an upstream component, but it does not neutralize or incorrectly neutralizes special elements that could modify the syntax or behavior of the intended code segment.'),
    'CWE-617': ('Reachable Assertion',
                'The product contains an assert() or similar statement that can be triggered by an attacker, which leads to an application exit or other behavior that is more severe than necessary.'
                'While assertion is good for catching logic errors and reducing the chances of reaching more serious vulnerability conditions, it can still lead to a denial of service.'
                'For example, if a server handles multiple simultaneous connections, and an assert() occurs in one single connection that causes all other connections to be dropped, this is a reachable assertion that leads to a denial of service.'),
    'CWE-327': ('Use of a Broken or Risky Cryptographic Algorithm',
                'The software uses a cryptographic algorithm that has known '
                'weaknesses (e.g. MD5, SHA1) making it unsuitable for security '
                'purposes such as password hashing, signing, or integrity checks.'),
}

def show_candidate(label, test_id, source):
    rows = df_issues[
        (df_issues['test_id'] == test_id) &
        (df_issues['source']  == source)
    ].head(1)
    if rows.empty:
        print(f'Candidate {label}: no finding for {test_id}/{source}')
        return

    row     = rows.iloc[0]
    task_id = 'HumanEval/' + row['file'].replace('HumanEval_', '')
    cwe     = row['cwe']
    cwe_def = CWE_DEFINITIONS.get(cwe, ('', ''))

    print(f'{"═"*65}')
    print(f'CANDIDATE {label}')
    print(f'{"═"*65}')
    print(f'  Bandit ID   : {row["test_id"]}')
    print(f'  CWE         : {cwe} — {cwe_def[0]}')
    print(f'  Definition  : {cwe_def[1]}')
    print(f'  Severity    : {row["severity"]} / Confidence: {row["confidence"]} / Priority: {row["priority"]}')
    print(f'  Flag message: {row["text"]}')
    print(f'  Source      : {row["source_label"]}')
    print()

    # Task prompt
    prompt = prompt_by_id.get(task_id, '')
    if prompt:
        lines = prompt.strip().split('\n')[:10]
        print('  TASK PROMPT (what the model was asked to implement):')
        print('  ' + '─'*57)
        for l in lines:
            print(f'  {l}')
        if len(prompt.strip().split('\n')) > 10:
            print('  ...')
        print('  ' + '─'*57)
    print()

    # Code context
    src = df[(df['task_id'] == task_id) & (df['source'] == source)]
    if not src.empty:
        code_lines = src.iloc[0]['code'].split('\n')
        start = max(0, int(row['line']) - 4)
        end   = min(len(code_lines), int(row['line']) + 3)
        print('  FLAGGED CODE (>>> marks the line Bandit flagged):')
        print('  ' + '─'*57)
        for j, line in enumerate(code_lines[start:end], start=start+1):
            marker = '>>>' if j == int(row['line']) else '   '
            print(f'  {marker} {j:>3}: {line}')
        print('  ' + '─'*57)
    print()


b307_row = df_issues[df_issues['test_id'] == 'B307'].iloc[0] if len(df_issues[df_issues['test_id'] == 'B307']) > 0 else None
b101_row = df_issues[df_issues['test_id'] == 'B101'].iloc[0] if len(df_issues[df_issues['test_id'] == 'B101']) > 0 else None
b324_row = df_issues[df_issues['test_id'] == 'B324'].iloc[0] if len(df_issues[df_issues['test_id'] == 'B324']) > 0 else None

if b307_row is not None:
    show_candidate('A', 'B307', b307_row['source'])
if b101_row is not None:
    show_candidate('B', 'B101', b101_row['source'])
if b324_row is not None:
    show_candidate('C', 'B324', b324_row['source'])

In [ ]:
# ── Your turn ──────────────────────────────────────────────────────────────
# Look at the 3 candidates above. For each one ask yourself:
#
#   1. What is the vulnerability type and why would it normally be dangerous?
#   2. What was the model actually ASKED to do by the task prompt?
#   3. Does the flagged code pattern represent a real security risk in this context?
#
# Then fill in your answers below:

YOUR_ANSWERS = {
    'A': {
        'verdict': '',    # 'true_positive' or 'false_positive'
        'reason':  '',    # one sentence explaining your thought process
    },
    'B': {
        'verdict': '',
        'reason':  '',
    },
    'C': {
        'verdict': '',
        'reason':  '',
    },
}

for candidate, ans in YOUR_ANSWERS.items():
    v = ans['verdict']
    r = ans['reason']
    if v and r:
        print(f'Candidate {candidate}: {v.upper()} — {r}')
    else:
        print(f'Candidate {candidate}: (not answered yet)')

## ‼️ Run the cell below without opening it! ‼️

In [ ]:
# @title
# ── Reveal ─────────────────────────────────────────────────────────────────

print("""
═══════════════════════════════════════════════════════════════════
REVEAL — False Positive Analysis
═══════════════════════════════════════════════════════════════════

CANDIDATE A — B307 / CWE-94 / Code Injection (eval)
──────────────────────────────────────────────────────
Verdict: TRUE POSITIVE

The task asks the model to evaluate an algebraic expression built
from two lists. The human solution builds the expression as a
string and calls eval() on it. This works — but eval() executes
arbitrary Python, not just arithmetic. In a real codebase, if
operator or operand ever came from user input, an attacker could
inject any Python expression and execute it.

Bandit is correct. The safer fix is to compute the result
iteratively using operator.add / operator.mul etc., or use
ast.literal_eval() which only handles literals. The task does not
require eval() — it is just the path of least resistance.

CANDIDATE B — B101 / CWE-617 / Reachable Assertion
──────────────────────────────────────────────────────
Verdict: TRUE POSITIVE

CodeLlama adds an assert to check that opening and closing
parentheses are balanced before processing the string. The CWE
definition explains the risk: if an attacker can trigger an
assert, it causes an application exit or behavior more severe
than necessary — in a server context, one malformed input string
could crash the entire connection handler, dropping all other
active connections as a denial of service.

Here, any caller passing an unbalanced string — whether by
mistake or deliberately — triggers the assert and abruptly
terminates execution. The function is supposed to parse
parenthesis groups, and unbalanced input is a completely
plausible thing a caller might pass. Bandit is correct: this
is a reachable assertion that an attacker can trigger with
ordinary input.

The fix is to replace the assert with an explicit raise that
handles the error gracefully rather than crashing:

  if paren_string.count("(") != paren_string.count(")"):
      raise ValueError("Unbalanced parentheses")

CANDIDATE C — B324 / CWE-327 / Weak Hash (MD5)
──────────────────────────────────────────────────────
Verdict: FALSE POSITIVE

CWE-327 applies when MD5 is used for password hashing, digital
signatures, or data integrity verification — contexts where
collision resistance matters. Here the task prompt explicitly asks
the model to return the MD5 hash of a string. MD5 IS the correct
answer. It is not used for any security purpose.

This fires on every single source — human and all 8 LLMs — for
the same task. That is the key signal: when a Bandit rule fires
uniformly across all sources on the same task, it is almost always
a false positive driven by the task itself, not by model quality.

The fix to silence it legitimately without changing logic:
  hashlib.md5(data, usedforsecurity=False)

═══════════════════════════════════════════════════════════════════
DISCUSSION QUESTIONS:

1. Are vulnerability rates higher in LLM code than human code?
   Look at per_100loc in the summary — what do you notice?

2. Does the vulnerability TYPE differ across models? Does GPT-4
   trigger different CWEs than CodeLlama or StarCoder?

3. Given B324 fires on 100% of sources for the same task, what
   does this tell us about using raw Bandit counts as a quality
   signal in LLM evaluation benchmarks?

4. How would you design a better pipeline that reduces false
   positives before comparing LLM vs. human security profiles?
═══════════════════════════════════════════════════════════════════
""")

---
## 03f ·  Findings

> Discussion: Are vulnerability rates higher in LLM code? Does the vulnerability type differ? What might explain this?

In [ ]:
human_density = df_summary[df_summary['source'] == 'human']['per_100loc'].values[0]
llm_df        = df_summary[df_summary['source'] != 'human']
most_vuln_llm  = llm_df.loc[llm_df['per_100loc'].idxmax()]
least_vuln_llm = llm_df.loc[llm_df['per_100loc'].idxmin()]

print('=' * 65)
print('FINDING #1 — Vulnerability Density: LLM vs. Human')
print('=' * 65)
print(f'  Human baseline:       {human_density:.2f} issues per 100 LOC')
print(f'  Most vulnerable LLM:  {most_vuln_llm["source_label"]} ({most_vuln_llm["per_100loc"]:.2f}/100 LOC)')
print(f'  Least vulnerable LLM: {least_vuln_llm["source_label"]} ({least_vuln_llm["per_100loc"]:.2f}/100 LOC)')
print()

print('=' * 65)
print('FINDING #2 — Critical Issues (HIGH sev + HIGH conf)')
print('=' * 65)
for _, row in df_summary.sort_values('critical', ascending=False).iterrows():
    bar = '█' * int(row['critical'])
    print(f"  {row['source_label']:>18}: {int(row['critical']):>3} critical  {bar}")
print()

print('=' * 65)
print('FINDING #3 — Top 3 Vulnerability Types')
print('=' * 65)
for tid, cwe, name in zip(top3_ids, top3_cwes, top3_names):
    count = vuln_counts[(tid, cwe, name)]
    print(f'  {tid} ({cwe}): {name} — {count} occurrences')
print()

print('=' * 65)
print('DISCUSSION:')
print('  Are vulnerability rates higher in LLM code?')
print('  Does the vulnerability TYPE differ across models?')
print('  What might explain these patterns?')
print('=' * 65)

## Section 04 · Integrated Analysis — Part 1 Wrap-Up

> ** ~15 minutes** | Section 02 (SonarCloud) and Section 03 (Bandit) are prerequisites!!

### Pipeline recap
**Code Artifacts → SonarCloud → Bandit → Unified Metrics → Visualization & Comparison**

We now combine both result sets into a single analysis view and compare our empirical findings against what prior studies found.

| Step | What we did | Output |
|------|-------------|--------|
| SonarCloud | Maintainability + reliability static analysis | Quality Analysis |
| Bandit | Security vulnerability scan | Security Analysis |
| **This section** | **Merge, visualize, compare to literature** | **Unified view** |

## 04a · Merge SonarCloud + Bandit into Unified Table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Merge on source key
df_sonar  = df_proj[['source', 'source_label', 'ncloc', 'code_smells',
                      'smells_per_100loc', 'sqale_rating_label',
                      'bugs', 'reliability_rating_label',
                      'cognitive_complexity']].copy()

df_bandit = df_summary[['source', 'total_issues', 'high_sev', 'critical',
                         'per_100loc']].copy()
df_bandit = df_bandit.rename(columns={
    'total_issues': 'bandit_total',
    'high_sev':     'bandit_high',
    'critical':     'bandit_critical',
    'per_100loc':   'bandit_per_100loc',
})

df_unified = df_sonar.merge(df_bandit, on='source')

DISPLAY = ['source_label', 'ncloc', 'code_smells', 'smells_per_100loc',
           'sqale_rating_label', 'cognitive_complexity',
           'bugs', 'reliability_rating_label',
           'bandit_total', 'bandit_high', 'bandit_critical', 'bandit_per_100loc']

print('📊 Unified Quality + Security Table')
print('=' * 195)
print(df_unified[DISPLAY].to_string(index=False))
print('=' * 195)
print()
print('Column guide:')
print('  smells_per_100loc    — SonarCloud code smells normalized by LOC')
print('  sqale_rating_label   — Maintainability A-E (A=best)')
print('  reliability_rating_label — Reliability A-E')
print('  bandit_per_100loc    — Bandit issues normalized by LOC')
print('  bandit_critical      — HIGH severity + HIGH confidence findings')

---
## 04b · Unified Heatmap — All Dimensions at Once

All metrics normalized

In [ ]:
SOURCE_ORDER = ['human', 'gpt4', 'chatgpt', 'codellama', 'starcoder',
                'mistral', 'codegen2b', 'codegen6b', 'codegen16b',]

# Metrics to include and whether higher = better or lower = better
METRICS_CONFIG = [
    ('smells_per_100loc',   'Smells/100 LOC',        'lower'),
    ('cognitive_complexity','Cognitive Complexity',   'lower'),
    ('bugs',                'Bugs (SonarCloud)',      'lower'),
    ('bandit_per_100loc',   'Vuln Density/100 LOC',  'lower'),
    ('bandit_critical',     'Critical Vulns',         'lower'),
    ('bandit_high',         'HIGH Sev Findings',     'lower'),
]

df_heat = df_unified.set_index('source').loc[SOURCE_ORDER].copy()
df_heat.index = [SOURCE_LABELS[s] for s in SOURCE_ORDER]

# Build normalized matrix
matrix = {}
for col, label, direction in METRICS_CONFIG:
    vals = df_heat[col].astype(float)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        normed = pd.Series([0.5] * len(vals), index=vals.index)
    else:
        normed = (vals - mn) / (mx - mn)
    # Flip so that for all metrics: 0 = best (green), 1 = worst (red)
    if direction == 'lower':
        normed = 1 - normed
    matrix[label] = normed

df_matrix = pd.DataFrame(matrix)

# Annotation values (raw numbers, not normalized)
annot = pd.DataFrame(index=df_matrix.index)
for col, label, _ in METRICS_CONFIG:
    annot[label] = df_heat[col].round(2).astype(str)

fig, ax = plt.subplots(figsize=(13, 7))
# green = better, red = worse — RdYlGn reversed for 'higher norm = better'
sns.heatmap(df_matrix, annot=annot, fmt='', cmap='RdYlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1,
            cbar_kws={'label': '← worse   /   better →'})
ax.set_title(
    'Unified Quality + Security Heatmap\n'
    'All metrics normalized',
    fontweight='bold', fontsize=13
)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/integrated_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved.')

---
## 04c · Radar Chart — Trade-offs Across All Dimensions

Each axis is normalized. A model with a large polygon is stronger across all dimensions.

In [ ]:
# Radar chart: 5 axes, one polygon per source
# Axes: Maintainability, Low Complexity, Reliability, Security, Low Vuln Density

RADAR_METRICS = [
    ('smells_per_100loc',   'Maintainability\n(fewer smells)',  'lower'),
    ('cognitive_complexity','Low Complexity',                    'lower'),
    ('bugs',                'Reliability\n(fewer bugs)',        'lower'),
    ('bandit_per_100loc',   'Security\n(fewer vulns)',          'lower'),
    ('bandit_critical',     'No Critical\nFindings',            'lower'),
]

labels_r = [m[1] for m in RADAR_METRICS]
N = len(labels_r)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

# Normalize each axis 0-1, higher = better
df_radar = df_unified.set_index('source').loc[SOURCE_ORDER].copy()
radar_data = {}
for col, label, direction in RADAR_METRICS:
    vals = df_radar[col].astype(float)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        normed = pd.Series([0.5]*len(vals), index=vals.index)
    else:
        normed = (vals - mn) / (mx - mn)
    if direction == 'lower':
        normed = 1 - normed
    radar_data[col] = normed
df_radar_norm = pd.DataFrame(radar_data, index=df_radar.index)

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

RADAR_SOURCES = ['human', 'gpt4', 'chatgpt', 'codellama', 'starcoder', 'mistral', 'codegen2b', 'codegen6b', 'codegen16b']
for source in RADAR_SOURCES:
    vals_r = df_radar_norm.loc[source, [m[0] for m in RADAR_METRICS]].tolist()
    vals_r += vals_r[:1]
    ax.plot(angles, vals_r, linewidth=2, label=SOURCE_LABELS[source],
            color=COLORS[source])
    ax.fill(angles, vals_r, alpha=0.07, color=COLORS[source])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], fontsize=7, color='grey')
ax.set_title(
    'Radar Chart: Quality + Security Profile\n'
    'Further from centre = better | Human + top LLMs',
    fontweight='bold', fontsize=12, pad=20
)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('/tmp/integrated_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Saved.')

---
## 04d · Compare Against Prior Literature

Three empirical findings from the slide — let's check what our data actually shows.

In [ ]:
# ── Finding 1: Molison et al. 2025 ──────────────────────────────────
# 'LLM code has 20-40% more code smells on average,
#  with higher cognitive complexity per function'

human_smells = df_unified[df_unified['source']=='human']['smells_per_100loc'].values[0]
human_cog    = df_unified[df_unified['source']=='human']['cognitive_complexity'].values[0]
llm_only     = df_unified[df_unified['source']!='human']
llm_smells_mean = llm_only['smells_per_100loc'].mean()
llm_cog_mean    = llm_only['cognitive_complexity'].mean()

smell_pct_diff = (llm_smells_mean - human_smells) / human_smells * 100 if human_smells > 0 else float('nan')
cog_pct_diff   = (llm_cog_mean - human_cog) / human_cog * 100 if human_cog > 0 else float('nan')

print('=' * 68)
print('LITERATURE FINDING 1 — Molison et al., 2025')
print('Claim: LLM code has 20-40% more code smells and higher')
print('       cognitive complexity per function')
print('=' * 68)
print(f'  Human smells/100 LOC   : {human_smells:.2f}')
print(f'  LLM avg smells/100 LOC : {llm_smells_mean:.2f}')
print(f'  Difference             : {smell_pct_diff:+.1f}%')
print()
if smell_pct_diff > 20:
    print('  ✅ CONSISTENT with Molison et al. — smells higher in LLM code')
elif smell_pct_diff > 0:
    print('  ⚠️  PARTIALLY consistent — LLM smells higher but below 20% threshold')
else:
    print('  ❌ NOT consistent — human code has more smells than LLM average')
print()

# ── Finding 3: Siddiq et al. 2024 ────────────────────────────────────
# 'LLMs generate more injection-prone code;

print('=' * 68)
print('LITERATURE FINDING 2 — Siddiq et al., 2024')
print('Claim: LLMs generate more injection-prone code;')
print('       human code clusters around CWE-330 weak randomness')
print('=' * 68)

# Count injection-related CWEs (CWE-94)
injection_cwes = ['CWE-78', 'CWE-89', 'CWE-94', 'CWE-094']
cwe330 = ['CWE-330', 'CWE-0330']

human_injection = len(df_issues[(df_issues['source']=='human') & (df_issues['cwe'].isin(injection_cwes))])
human_cwe330    = len(df_issues[(df_issues['source']=='human') & (df_issues['cwe'].isin(cwe330))])
llm_injection   = len(df_issues[(df_issues['source']!='human') & (df_issues['cwe'].isin(injection_cwes))])
llm_cwe330      = len(df_issues[(df_issues['source']!='human') & (df_issues['cwe'].isin(cwe330))])

print(f'  Human — injection-related (CWE-94) : {human_injection}')
print(f'  Starcoder   — injection-related (CWE-94) : 4')
print(f'  CodeLLaMA-34B   — injection-related (CWE-94) : 2')
print()



if llm_injection > human_injection:
    print('  ✅ CONSISTENT — LLMs show more injection-related findings')
else:
    print('  ❌ NOT consistent with this claim in our dataset')


---
## 04e · Your Reflection

Before the discussion, note your own observations from the data.

In [ ]:
YOUR_REFLECTION = {
    'finding_that_you_may_have_expected': '',
    # Anything you found consistent from your own experiences using LLMs?

    'finding_that_surprised_you': '',
    # What did you NOT expect from the results?
    # Could be something that contradicted the literature,
    # or something unexpected within the models.

    'model_you_would_use_in_production': '',
    # Based purely on this analysis — which model would you
    # be most comfortable deploying? Why?
}

print('Fill in YOUR_REFLECTION above, then share with the group.')
print()
for k, v in YOUR_REFLECTION.items():
    print(f'{k}:')
    print(f'  {v if v else "(not filled in yet)"}')
    print()

In [ ]:
print('=' * 68)
print('PART 1 WRAP-UP — What We Found')
print('=' * 68)

# Best and worst per dimension
du = df_unified.copy()
best_maint   = du.loc[du['smells_per_100loc'].idxmin(), 'source_label']
worst_maint  = du.loc[du['smells_per_100loc'].idxmax(), 'source_label']
best_cog     = du.loc[du['cognitive_complexity'].idxmin(), 'source_label']
worst_cog    = du.loc[du['cognitive_complexity'].idxmax(), 'source_label']
best_sec     = du.loc[du['bandit_per_100loc'].idxmin(), 'source_label']
worst_sec    = du.loc[du['bandit_per_100loc'].idxmax(), 'source_label']
best_rel     = du.loc[du['bugs'].idxmin(), 'source_label']

print()
print('  MAINTAINABILITY (SonarCloud smells/100 LOC)')
print(f'    Best  : {best_maint}')
print(f'    Worst : {worst_maint}')
print()
print('  COGNITIVE COMPLEXITY')
print(f'    Simplest code : {best_cog}')
print(f'    Most complex  : {worst_cog}')
print()
print('  RELIABILITY (SonarCloud bugs)')
print(f'    Fewest bugs   : {best_rel}')
print()
print('  SECURITY (Bandit issues/100 LOC)')
print(f'    Best  : {best_sec}')
print(f'    Worst : {worst_sec}')
print()
print('  FALSE POSITIVE RATE')
print('    B324 (MD5 hashlib) fired on 100% of sources for one task.')
print('    Raw Bandit counts require human review before use as a')
print('    quality signal in LLM evaluation benchmarks.')
print()

---
# Section 5 - Plug & Play - Use these tools on your own repos!


##05 - Scan Your Own Repos: Bandit + CodeQL Security Scan

In [ ]:

REPO_URL = "https://github.com/jessica-po/FSECodeQLandBanditPlugandPlayExample.git"  # <-- replace this
REPO_DIR = "target_repo"
REPORT_DIR = "security_reports"

!rm -rf $REPO_DIR $REPORT_DIR codeql-db
!mkdir -p $REPORT_DIR

!git clone $REPO_URL $REPO_DIR


# 05a - Install bandit

In [ ]:
!pip install bandit -q

In [ ]:
!bandit -r $REPO_DIR -f json -o $REPORT_DIR/bandit_report.json || true

#05b - Install CodeQL

In [ ]:
# Install CodeQL CLI
!wget -q https://github.com/github/codeql-cli-binaries/releases/latest/download/codeql-linux64.zip
!unzip -q codeql-linux64.zip
!./codeql/codeql version

##05c - Create CodeQL database for Python

In [ ]:
!./codeql/codeql database create codeql-db \
  --language=python \
  --source-root=$REPO_DIR \
  --build-mode=none \
  --overwrite

##05d - Download Python CodeQL Query Pack and Run CodeQL Analysis

In [ ]:
# Download the Python CodeQL query pack
!./codeql/codeql pack download codeql/python-queries

# Run CodeQL analysis
!./codeql/codeql database analyze codeql-db \
  codeql/python-queries:codeql-suites/python-security-and-quality.qls \
  --format=sarif-latest \
  --output=$REPORT_DIR/codeql_report.sarif \
  --sarif-add-snippets || true

##05e - Generate Reports


In [ ]:
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

report_dir = Path("security_reports")
repo_dir = Path("target_repo")

rows = []

def get_snippet(file_path, line_number, context=2):
    if not file_path or not line_number:
        return ""

    path = repo_dir / file_path
    if not path.exists():
        path = Path(file_path)

    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    start = max(int(line_number) - context - 1, 0)
    end = min(int(line_number) + context, len(lines))

    snippet = []
    for i in range(start, end):
        marker = ">>" if i + 1 == int(line_number) else "  "
        snippet.append(f"{marker} {i + 1}: {lines[i]}")
    return "\n".join(snippet)

def categorize_codeql(tags, rule_id, description):
    text = " ".join(tags + [rule_id, description]).lower()

    if "security" in text or "cwe" in text:
        return "Security"
    if "maintainability" in text or "readability" in text or "unused" in text:
        return "Code smell / maintainability"
    if "correctness" in text or "bug" in text or "error" in text:
        return "Bug / correctness"
    if "performance" in text:
        return "Performance"
    return "Quality / other"

def explain_category(category):
    explanations = {
        "Security": "Potential vulnerability or unsafe data flow.",
        "Bug / correctness": "Code may behave incorrectly or fail in some cases.",
        "Code smell / maintainability": "Code is valid but may be unused, confusing, or harder to maintain.",
        "Performance": "Code may be inefficient or unnecessarily expensive.",
        "Quality / other": "General quality issue that should be reviewed manually."
    }
    return explanations.get(category, "Review manually.")

# -----------------------
# Bandit
# -----------------------
bandit_path = report_dir / "bandit_report.json"

if bandit_path.exists():
    with open(bandit_path, "r") as f:
        bandit_data = json.load(f)

    for item in bandit_data.get("results", []):
        file_path = item.get("filename")
        line = item.get("line_number")

        rows.append({
            "tool": "Bandit",
            "category": "Security",
            "file": file_path,
            "line": line,
            "rule_id": item.get("test_id"),
            "rule_name": item.get("test_name"),
            "severity": item.get("issue_severity"),
            "confidence": item.get("issue_confidence"),
            "message": item.get("issue_text"),
            "what_it_means": "Bandit flagged a Python security pattern.",
            "snippet": get_snippet(file_path, line)
        })

# -----------------------
# CodeQL
# -----------------------
codeql_path = report_dir / "codeql_report.sarif"

if codeql_path.exists():
    with open(codeql_path, "r") as f:
        codeql_data = json.load(f)

    for run in codeql_data.get("runs", []):
        rules = {}

        for rule in run.get("tool", {}).get("driver", {}).get("rules", []):
            rule_id = rule.get("id")
            properties = rule.get("properties", {})

            rules[rule_id] = {
                "name": rule.get("name", ""),
                "description": rule.get("shortDescription", {}).get("text", ""),
                "full_description": rule.get("fullDescription", {}).get("text", ""),
                "tags": properties.get("tags", []),
                "precision": properties.get("precision", ""),
                "problem_severity": properties.get("problem.severity", "")
            }

        for result in run.get("results", []):
            rule_id = result.get("ruleId")
            rule_info = rules.get(rule_id, {})

            location = result.get("locations", [{}])[0].get("physicalLocation", {})
            artifact = location.get("artifactLocation", {})
            region = location.get("region", {})

            file_path = artifact.get("uri")
            line = region.get("startLine")

            description = rule_info.get("description", "")
            tags = rule_info.get("tags", [])
            category = categorize_codeql(tags, rule_id, description)

            rows.append({
                "tool": "CodeQL",
                "category": category,
                "file": file_path,
                "line": line,
                "rule_id": rule_id,
                "rule_name": rule_info.get("name", ""),
                "severity": rule_info.get("problem_severity", result.get("level", "warning")),
                "confidence": rule_info.get("precision", ""),
                "message": result.get("message", {}).get("text"),
                "what_it_means": explain_category(category),
                "snippet": get_snippet(file_path, line)
            })

report_df = pd.DataFrame(rows)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

display(report_df)

report_df.to_csv(report_dir / "combined_security_quality_report.csv", index=False)

##05f - Final Plug and Play Summary

In [ ]:
if report_df.empty:
    print("No findings found.")
else:
    print("Findings by tool and category")
    summary = (
        report_df
        .groupby(["tool", "category"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(summary)

    print("Most common rules")
    rule_summary = (
        report_df
        .groupby(["tool", "rule_id", "rule_name", "category"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(rule_summary)